# Chapter 39 — Learning to Predict Cell Divisions

This lesson replaces the Chapter 38 heuristic division score with a supervised classifier.

## Objectives

- Load Chapter 38 division candidates
- Label candidates from GEFF ground truth
- Audit the severe class imbalance
- Train LightGBM when enough positive examples exist
- Use grouped cross-validation by parent track
- Select a probability threshold
- Enforce non-conflicting lineage constraints
- Save predictions, metrics, model features, and the trained model

> The current sample contains only one GT division. The notebook therefore includes a safe fallback when there are not enough positive candidates to train a meaningful model.


In [1]:
!pip install -q lightgbm joblib zarr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 64.4 MB/s eta 0:00:00


In [2]:
from __future__ import annotations

from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from IPython.display import display

warnings.filterwarnings("ignore")


In [3]:
@dataclass(frozen=True)
class Config:
    sample_id: str = "44b6_12dfb391"
    random_state: int = 42
    minimum_positive_examples: int = 2
    minimum_negative_examples: int = 10
    minimum_track_match_purity: float = 0.50
    minimum_track_match_count: int = 1
    default_threshold: float = 0.50
    minimum_precision_for_threshold: float = 0.20
    maximum_cv_splits: int = 5


CONFIG = Config()
print(CONFIG)


Config(sample_id='44b6_12dfb391', random_state=42, minimum_positive_examples=2, minimum_negative_examples=10, minimum_track_match_purity=0.5, minimum_track_match_count=1, default_threshold=0.5, minimum_precision_for_threshold=0.2, maximum_cv_splits=5)


## 1. Load Chapter 38 outputs

In [4]:
DATA_DIR = Path("/kaggle/input/datasets/emailchrismathews/data38")

paths = {
    "candidates": DATA_DIR / "/kaggle/input/datasets/emailchrismathews/data38/chapter38_division_candidates.csv",
    "selected": DATA_DIR / "/kaggle/input/datasets/emailchrismathews/data38/chapter38_selected_divisions.csv",
    "track_to_gt": DATA_DIR / "/kaggle/input/datasets/emailchrismathews/data38/chapter38_track_to_gt_mapping.csv",
}

for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")
    print(f"{name}: {path}")

candidates_df = pd.read_csv(paths["candidates"])
heuristic_selected_df = pd.read_csv(paths["selected"])
track_to_gt_df = pd.read_csv(paths["track_to_gt"])

print("candidates", len(candidates_df))
print("heuristic selections", len(heuristic_selected_df))
print("track mappings", len(track_to_gt_df))

display(candidates_df.head())
display(track_to_gt_df.head())


candidates: /kaggle/input/datasets/emailchrismathews/data38/chapter38_division_candidates.csv
selected: /kaggle/input/datasets/emailchrismathews/data38/chapter38_selected_divisions.csv
track_to_gt: /kaggle/input/datasets/emailchrismathews/data38/chapter38_track_to_gt_mapping.csv
candidates 93
heuristic selections 13
track mappings 139


,parent_track_id,daughter_track_id_a,daughter_track_id_b,parent_end_t,daughter_start_t_a,daughter_start_t_b,frame_gap_a,frame_gap_b,distance_a,distance_b,...,daughter_mean_rank_b,parent_mean_pred_dist,daughter_mean_pred_dist_a,daughter_mean_pred_dist_b,distance_score,separation_score,balance_score,parent_quality_score,daughter_quality_score,division_score
0,83,155,156,14,15,15,1,1,9.911194,8.797476,...,58.882353,14.544073,13.405217,15.525601,0.740157,0.292533,0.679109,0.401419,0.552195,0.581799
1,349,389,390,61,62,62,1,1,11.440104,9.272829,...,43.400000,15.386548,16.003807,15.685215,0.712320,0.725038,0.555277,0.333126,0.312078,0.565904
2,108,181,185,17,20,20,3,3,27.948702,25.654936,...,80.666667,15.214449,15.726143,16.393109,0.255505,0.839483,0.958965,0.415234,0.511296,0.546122
3,108,185,188,17,20,20,3,3,25.654936,29.880785,...,77.791667,15.214449,16.393109,16.050463,0.228671,0.960825,0.804288,0.415234,0.555268,0.530591
4,208,318,365,56,58,57,2,1,17.525345,11.261994,...,96.000000,16.674182,16.271588,15.681017,0.600176,0.990664,0.521306,0.247673,0.151152,0.522746


,track_id,gt_node_id,match_count,total_matches,track_match_purity
0,0,1.060000e+11,1,2,0.500000
1,1,1.060000e+11,1,2,0.500000
2,3,1.060000e+11,1,6,0.166667
3,4,1.180000e+11,1,5,0.200000
4,5,1.060000e+11,1,2,0.500000


## 2. Load GEFF divisions

In [5]:
def find_geff(sample_id):
    expected = Path(
        "/kaggle/input/competitions/"
        "biohub-cell-tracking-during-development/train"
    ) / f"{sample_id}.geff"

    if expected.exists():
        return expected

    matches = list(Path("/kaggle/input").rglob(f"{sample_id}.geff"))

    if matches:
        return sorted(matches, key=lambda p: len(str(p)))[0]

    raise FileNotFoundError(f"Could not find {sample_id}.geff")


def load_gt_divisions(sample_id):
    geff_path = find_geff(sample_id)
    print("GEFF:", geff_path)

    geff = zarr.open(geff_path, mode="r")
    edge_ids = np.asarray(geff["edges/ids"][:])

    edges = pd.DataFrame({
        "gt_source": edge_ids[:, 0],
        "gt_target": edge_ids[:, 1],
    })

    rows = []

    for parent, group in edges.groupby("gt_source", sort=True):
        daughters = sorted(group["gt_target"].unique())

        for daughter_a, daughter_b in combinations(daughters, 2):
            rows.append({
                "gt_parent": parent,
                "gt_daughter_a": daughter_a,
                "gt_daughter_b": daughter_b,
            })

    return pd.DataFrame(rows)


gt_divisions_df = load_gt_divisions(CONFIG.sample_id)

print("GT divisions", len(gt_divisions_df))
display(gt_divisions_df)


GEFF: /kaggle/input/competitions/biohub-cell-tracking-during-development/train/44b6_12dfb391.geff
GT divisions 1


,gt_parent,gt_daughter_a,gt_daughter_b
0,172000000050,173000000050,173000000051


## 3. Label candidates using reliable track identities

In [6]:
def label_candidates(candidates, mapping_table, gt_divisions, config=CONFIG):
    reliable = mapping_table[
        (mapping_table["track_match_purity"] >= config.minimum_track_match_purity)
        & (mapping_table["match_count"] >= config.minimum_track_match_count)
    ].copy()

    mapping = dict(zip(
        reliable["track_id"].astype(int),
        reliable["gt_node_id"],
    ))

    true_divisions = {
        (
            row.gt_parent,
            frozenset([row.gt_daughter_a, row.gt_daughter_b]),
        )
        for row in gt_divisions.itertuples(index=False)
    }

    result = candidates.copy()
    labels = []
    statuses = []
    mapped_parent = []
    mapped_a = []
    mapped_b = []

    for row in result.itertuples(index=False):
        parent = mapping.get(int(row.parent_track_id))
        daughter_a = mapping.get(int(row.daughter_track_id_a))
        daughter_b = mapping.get(int(row.daughter_track_id_b))

        mapped_parent.append(parent)
        mapped_a.append(daughter_a)
        mapped_b.append(daughter_b)

        all_reliable = (
            parent is not None
            and daughter_a is not None
            and daughter_b is not None
        )

        if not all_reliable:
            labels.append(np.nan)
            statuses.append("unlabeled")
            continue

        positive = (
            parent,
            frozenset([daughter_a, daughter_b]),
        ) in true_divisions

        labels.append(int(positive))
        statuses.append("positive" if positive else "negative")

    result["mapped_gt_parent"] = mapped_parent
    result["mapped_gt_daughter_a"] = mapped_a
    result["mapped_gt_daughter_b"] = mapped_b
    result["division_label"] = labels
    result["label_status"] = statuses

    print("Reliable mappings:", len(reliable))
    return result


labeled_df = label_candidates(
    candidates_df,
    track_to_gt_df,
    gt_divisions_df,
)

display(labeled_df["label_status"].value_counts(dropna=False).to_frame())
display(labeled_df.head())


Reliable mappings: 97


,count
label_status,
unlabeled,93


,parent_track_id,daughter_track_id_a,daughter_track_id_b,parent_end_t,daughter_start_t_a,daughter_start_t_b,frame_gap_a,frame_gap_b,distance_a,distance_b,...,separation_score,balance_score,parent_quality_score,daughter_quality_score,division_score,mapped_gt_parent,mapped_gt_daughter_a,mapped_gt_daughter_b,division_label,label_status
0,83,155,156,14,15,15,1,1,9.911194,8.797476,...,0.292533,0.679109,0.401419,0.552195,0.581799,NaN,NaN,NaN,NaN,unlabeled
1,349,389,390,61,62,62,1,1,11.440104,9.272829,...,0.725038,0.555277,0.333126,0.312078,0.565904,NaN,NaN,NaN,NaN,unlabeled
2,108,181,185,17,20,20,3,3,27.948702,25.654936,...,0.839483,0.958965,0.415234,0.511296,0.546122,NaN,1.260000e+11,NaN,NaN,unlabeled
3,108,185,188,17,20,20,3,3,25.654936,29.880785,...,0.960825,0.804288,0.415234,0.555268,0.530591,NaN,NaN,1.430000e+11,NaN,unlabeled
4,208,318,365,56,58,57,2,1,17.525345,11.261994,...,0.990664,0.521306,0.247673,0.151152,0.522746,NaN,NaN,NaN,NaN,unlabeled


## 4. Audit class balance

In [7]:
positive_count = int((labeled_df["division_label"] == 1).sum())
negative_count = int((labeled_df["division_label"] == 0).sum())
labeled_count = int(labeled_df["division_label"].notna().sum())

audit_df = pd.DataFrame({
    "metric": [
        "total_candidates",
        "labeled_candidates",
        "unlabeled_candidates",
        "positive_candidates",
        "negative_candidates",
        "positive_fraction",
    ],
    "value": [
        len(labeled_df),
        labeled_count,
        len(labeled_df) - labeled_count,
        positive_count,
        negative_count,
        positive_count / max(labeled_count, 1),
    ],
})

display(audit_df)

can_train = (
    positive_count >= CONFIG.minimum_positive_examples
    and negative_count >= CONFIG.minimum_negative_examples
)

print("Can train supervised model:", can_train)

if not can_train:
    print(
        "Not enough labeled positives. The notebook will use a clearly "
        "marked heuristic fallback while still saving the full training dataset."
    )


,metric,value
0,total_candidates,93.0
1,labeled_candidates,0.0
2,unlabeled_candidates,93.0
3,positive_candidates,0.0
4,negative_candidates,0.0
5,positive_fraction,0.0


Can train supervised model: False
Not enough labeled positives. The notebook will use a clearly marked heuristic fallback while still saving the full training dataset.


## 5. Define model features

In [8]:
FEATURE_COLUMNS = [
    "frame_gap_a",
    "frame_gap_b",
    "distance_a",
    "distance_b",
    "mean_distance",
    "distance_balance",
    "daughter_separation",
    "daughter_length_balance",
    "parent_num_detections",
    "daughter_num_detections_a",
    "daughter_num_detections_b",
    "parent_mean_rank",
    "daughter_mean_rank_a",
    "daughter_mean_rank_b",
    "parent_mean_pred_dist",
    "daughter_mean_pred_dist_a",
    "daughter_mean_pred_dist_b",
    "distance_score",
    "separation_score",
    "balance_score",
    "parent_quality_score",
    "daughter_quality_score",
]

missing = [c for c in FEATURE_COLUMNS if c not in labeled_df.columns]

if missing:
    raise ValueError(f"Missing features: {missing}")

display(labeled_df[FEATURE_COLUMNS].describe().T)


,count,mean,std,min,25%,50%,75%,max
frame_gap_a,93.0,2.655914,0.541628,1.000000,2.000000,3.000000,3.000000,3.000000
frame_gap_b,93.0,2.720430,0.518476,1.000000,3.000000,3.000000,3.000000,3.000000
distance_a,93.0,26.998642,6.074328,9.422898,21.781216,27.948702,32.349850,35.952553
distance_b,93.0,27.796343,6.157800,8.797476,24.344166,29.264103,32.349850,35.881329
mean_distance,93.0,27.397493,5.856575,9.354335,24.223204,28.661045,31.855729,35.875496
distance_balance,93.0,0.904354,0.084487,0.561896,0.877423,0.929414,0.962058,0.999642
daughter_separation,93.0,7.251126,4.667010,1.284675,3.067120,5.915089,10.444655,17.586449
daughter_length_balance,93.0,0.539585,0.266128,0.071429,0.300000,0.500000,0.750000,1.000000
parent_num_detections,93.0,8.236559,5.842866,3.000000,3.000000,8.000000,8.000000,30.000000
daughter_num_detections_a,93.0,9.720430,7.494006,2.000000,4.000000,7.000000,18.000000,42.000000


## 6. Build LightGBM with a safe fallback

In [9]:
def make_model(random_state=42):
    try:
        from lightgbm import LGBMClassifier

        classifier = LGBMClassifier(
            objective="binary",
            n_estimators=300,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=10,
            subsample=0.85,
            colsample_bytree=0.85,
            class_weight="balanced",
            random_state=random_state,
            verbosity=-1,
        )
        model_name = "LightGBM"

    except Exception:
        classifier = ExtraTreesClassifier(
            n_estimators=400,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=random_state,
            n_jobs=-1,
        )
        model_name = "ExtraTrees"

    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", classifier),
    ])

    return pipeline, model_name


base_model, model_name = make_model(CONFIG.random_state)
print("Model:", model_name)


Model: LightGBM


## 7. Grouped cross-validation by parent track

In [10]:
def metrics_from_probabilities(y_true, probabilities, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    predictions = (np.asarray(probabilities) >= threshold).astype(int)

    result = {
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "average_precision": np.nan,
        "roc_auc": np.nan,
    }

    if len(np.unique(y_true)) == 2:
        result["average_precision"] = average_precision_score(
            y_true,
            probabilities,
        )
        result["roc_auc"] = roc_auc_score(
            y_true,
            probabilities,
        )

    return result


def grouped_cv(data, model):
    training = data[data["division_label"].notna()].copy()

    if training.empty or training["division_label"].nunique() < 2:
        return pd.DataFrame(), pd.DataFrame()

    groups = training["parent_track_id"].astype(int)
    unique_groups = groups.nunique()

    if unique_groups < 3:
        return pd.DataFrame(), pd.DataFrame()

    n_splits = min(CONFIG.maximum_cv_splits, unique_groups)
    splitter = GroupKFold(n_splits=n_splits)

    prediction_rows = []
    metric_rows = []

    X = training[FEATURE_COLUMNS]
    y = training["division_label"].astype(int)

    for fold, (train_index, valid_index) in enumerate(
        splitter.split(X, y, groups),
        start=1,
    ):
        y_train = y.iloc[train_index]
        y_valid = y.iloc[valid_index]

        if y_train.nunique() < 2:
            continue

        fold_model = clone(model)
        fold_model.fit(X.iloc[train_index], y_train)

        probabilities = fold_model.predict_proba(
            X.iloc[valid_index]
        )[:, 1]

        fold_metrics = metrics_from_probabilities(
            y_valid,
            probabilities,
        )
        fold_metrics["fold"] = fold
        fold_metrics["validation_rows"] = len(valid_index)
        fold_metrics["validation_positives"] = int(y_valid.sum())
        metric_rows.append(fold_metrics)

        fold_predictions = training.iloc[valid_index][[
            "parent_track_id",
            "daughter_track_id_a",
            "daughter_track_id_b",
            "division_label",
        ]].copy()

        fold_predictions["division_probability"] = probabilities
        fold_predictions["fold"] = fold
        prediction_rows.append(fold_predictions)

    if not prediction_rows:
        return pd.DataFrame(), pd.DataFrame()

    return (
        pd.concat(prediction_rows, ignore_index=True),
        pd.DataFrame(metric_rows),
    )


if can_train:
    cv_predictions_df, cv_metrics_df = grouped_cv(labeled_df, base_model)
else:
    cv_predictions_df = pd.DataFrame()
    cv_metrics_df = pd.DataFrame()

if cv_metrics_df.empty:
    print("Cross-validation skipped: insufficient class or group coverage.")
else:
    display(cv_metrics_df)
    display(cv_metrics_df.mean(numeric_only=True).to_frame("mean").T)


Cross-validation skipped: insufficient class or group coverage.


## 8. Tune the probability threshold

In [11]:
def choose_threshold(y_true, probabilities):
    if len(np.unique(y_true)) < 2:
        return CONFIG.default_threshold, pd.DataFrame()

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        probabilities,
    )

    rows = []

    for i, threshold in enumerate(thresholds):
        p = float(precision[i])
        r = float(recall[i])
        f1 = 2 * p * r / (p + r) if p + r else 0.0

        rows.append({
            "threshold": float(threshold),
            "precision": p,
            "recall": r,
            "f1": f1,
        })

    table = pd.DataFrame(rows)
    eligible = table[
        table["precision"] >= CONFIG.minimum_precision_for_threshold
    ]

    pool = eligible if not eligible.empty else table
    best = pool.sort_values(["f1", "recall"], ascending=False).iloc[0]

    return float(best["threshold"]), table


if not cv_predictions_df.empty:
    selected_threshold, threshold_search_df = choose_threshold(
        cv_predictions_df["division_label"].astype(int),
        cv_predictions_df["division_probability"],
    )
else:
    selected_threshold = CONFIG.default_threshold
    threshold_search_df = pd.DataFrame()

print("Selected threshold:", selected_threshold)


Selected threshold: 0.5


In [12]:
if not threshold_search_df.empty:
    plt.figure(figsize=(9, 5))
    plt.plot(
        threshold_search_df["threshold"],
        threshold_search_df["precision"],
        label="precision",
    )
    plt.plot(
        threshold_search_df["threshold"],
        threshold_search_df["recall"],
        label="recall",
    )
    plt.plot(
        threshold_search_df["threshold"],
        threshold_search_df["f1"],
        label="F1",
    )
    plt.axvline(selected_threshold, linestyle="--", label="selected")
    plt.xlabel("probability threshold")
    plt.ylabel("metric")
    plt.title("Threshold Search")
    plt.legend()
    plt.show()


## 9. Fit the final model or use the safe fallback

In [13]:
final_model = None

if can_train:
    training_df = labeled_df[labeled_df["division_label"].notna()].copy()

    final_model = clone(base_model)
    final_model.fit(
        training_df[FEATURE_COLUMNS],
        training_df["division_label"].astype(int),
    )

    labeled_df["learned_division_probability"] = final_model.predict_proba(
        labeled_df[FEATURE_COLUMNS]
    )[:, 1]

    prediction_source = f"trained_{model_name.lower()}"

else:
    score = labeled_df["division_score"].astype(float)

    if score.max() > score.min():
        probability = (score - score.min()) / (score.max() - score.min())
    else:
        probability = pd.Series(
            np.full(len(score), 0.5),
            index=score.index,
        )

    labeled_df["learned_division_probability"] = probability
    prediction_source = "heuristic_fallback_insufficient_labels"

print("Prediction source:", prediction_source)

display(
    labeled_df.sort_values(
        "learned_division_probability",
        ascending=False,
    )[[
        "parent_track_id",
        "daughter_track_id_a",
        "daughter_track_id_b",
        "division_score",
        "learned_division_probability",
        "division_label",
    ]].head(30)
)


Prediction source: heuristic_fallback_insufficient_labels


,parent_track_id,daughter_track_id_a,daughter_track_id_b,division_score,learned_division_probability,division_label
0,83,155,156,0.581799,1.000000,NaN
1,349,389,390,0.565904,0.951906,NaN
2,108,181,185,0.546122,0.892051,NaN
3,108,185,188,0.530591,0.845061,NaN
4,208,318,365,0.522746,0.821324,NaN
5,418,515,524,0.517694,0.806039,NaN
6,66,165,166,0.511626,0.787677,NaN
7,366,392,393,0.502771,0.760885,NaN
8,69,181,185,0.491435,0.726587,NaN
9,69,185,188,0.489090,0.719491,NaN


## 10. Inspect feature importance

In [14]:
feature_importance_df = pd.DataFrame()

if final_model is not None:
    classifier = final_model.named_steps["classifier"]

    if hasattr(classifier, "feature_importances_"):
        feature_importance_df = (
            pd.DataFrame({
                "feature": FEATURE_COLUMNS,
                "importance": classifier.feature_importances_,
            })
            .sort_values("importance", ascending=False)
            .reset_index(drop=True)
        )

if feature_importance_df.empty:
    print("Feature importance unavailable because no supervised model was trained.")
else:
    display(feature_importance_df)

    top = feature_importance_df.head(15)

    plt.figure(figsize=(10, 7))
    plt.barh(top["feature"][::-1], top["importance"][::-1])
    plt.xlabel("importance")
    plt.ylabel("feature")
    plt.title(f"{model_name} Feature Importance")
    plt.show()


Feature importance unavailable because no supervised model was trained.


## 11. Select non-conflicting learned divisions

In [15]:
def select_non_conflicting(candidates, threshold):
    ordered = candidates.sort_values(
        "learned_division_probability",
        ascending=False,
    )

    selected = []
    used_parents = set()
    used_daughters = set()

    for row in ordered.itertuples(index=False):
        if row.learned_division_probability < threshold:
            continue

        parent = int(row.parent_track_id)
        daughter_a = int(row.daughter_track_id_a)
        daughter_b = int(row.daughter_track_id_b)

        if parent in used_parents:
            continue

        if daughter_a in used_daughters or daughter_b in used_daughters:
            continue

        if daughter_a == daughter_b:
            continue

        if parent in {daughter_a, daughter_b}:
            continue

        selected.append(row._asdict())
        used_parents.add(parent)
        used_daughters.update([daughter_a, daughter_b])

    return pd.DataFrame(selected)


learned_selected_df = select_non_conflicting(
    labeled_df,
    selected_threshold,
)

print("Learned selected divisions:", len(learned_selected_df))

display(
    learned_selected_df[[
        "parent_track_id",
        "daughter_track_id_a",
        "daughter_track_id_b",
        "division_score",
        "learned_division_probability",
        "division_label",
    ]].head(30)
    if not learned_selected_df.empty
    else learned_selected_df
)


Learned selected divisions: 18


,parent_track_id,daughter_track_id_a,daughter_track_id_b,division_score,learned_division_probability,division_label
0,83,155,156,0.581799,1.000000,NaN
1,349,389,390,0.565904,0.951906,NaN
2,108,181,185,0.546122,0.892051,NaN
3,208,318,365,0.522746,0.821324,NaN
4,418,515,524,0.517694,0.806039,NaN
5,66,165,166,0.511626,0.787677,NaN
6,366,392,393,0.502771,0.760885,NaN
7,329,370,399,0.469573,0.660440,NaN
8,405,509,511,0.464687,0.645655,NaN
9,96,183,184,0.463125,0.640929,NaN


## 12. Compare learned and heuristic selections

In [16]:
def keys(table):
    if table.empty:
        return set()

    return {
        (
            int(row.parent_track_id),
            frozenset([
                int(row.daughter_track_id_a),
                int(row.daughter_track_id_b),
            ]),
        )
        for row in table.itertuples(index=False)
    }


heuristic_keys = keys(heuristic_selected_df)
learned_keys = keys(learned_selected_df)

comparison_df = pd.DataFrame({
    "metric": [
        "heuristic_selected",
        "learned_selected",
        "selected_by_both",
        "learned_only",
        "heuristic_only",
    ],
    "value": [
        len(heuristic_keys),
        len(learned_keys),
        len(heuristic_keys & learned_keys),
        len(learned_keys - heuristic_keys),
        len(heuristic_keys - learned_keys),
    ],
})

display(comparison_df)


,metric,value
0,heuristic_selected,13
1,learned_selected,18
2,selected_by_both,13
3,learned_only,5
4,heuristic_only,0


## 13. Evaluate final probabilities

In [17]:
evaluation_df = labeled_df[labeled_df["division_label"].notna()].copy()

if (
    not evaluation_df.empty
    and evaluation_df["division_label"].nunique() == 2
):
    metrics = metrics_from_probabilities(
        evaluation_df["division_label"].astype(int),
        evaluation_df["learned_division_probability"],
        selected_threshold,
    )

    final_metrics_df = pd.DataFrame([{
        **metrics,
        "threshold": selected_threshold,
        "prediction_source": prediction_source,
        "labeled_rows": len(evaluation_df),
        "positive_rows": int(evaluation_df["division_label"].sum()),
    }])

    predictions = (
        evaluation_df["learned_division_probability"]
        >= selected_threshold
    ).astype(int)

    confusion_df = pd.DataFrame(
        confusion_matrix(
            evaluation_df["division_label"].astype(int),
            predictions,
            labels=[0, 1],
        ),
        index=["actual_negative", "actual_positive"],
        columns=["predicted_negative", "predicted_positive"],
    )

    display(final_metrics_df)
    display(confusion_df)

else:
    final_metrics_df = pd.DataFrame([{
        "average_precision": np.nan,
        "roc_auc": np.nan,
        "precision": np.nan,
        "recall": np.nan,
        "f1": np.nan,
        "threshold": selected_threshold,
        "prediction_source": prediction_source,
        "labeled_rows": len(evaluation_df),
        "positive_rows": positive_count,
        "note": "Both positive and negative labels are required.",
    }])

    display(final_metrics_df)


,average_precision,roc_auc,precision,recall,f1,threshold,prediction_source,labeled_rows,positive_rows,note
0,NaN,NaN,NaN,NaN,NaN,0.5,heuristic_fallback_insufficient_labels,0,0,Both positive and negative labels are required.


## 14. Validate output constraints

In [18]:
def validate_selected(selected):
    if selected.empty:
        return pd.DataFrame([{
            "check": "selection",
            "passed": True,
            "details": "No candidates exceeded the threshold.",
        }])

    daughter_ids = pd.concat([
        selected["daughter_track_id_a"],
        selected["daughter_track_id_b"],
    ], ignore_index=True)

    checks = pd.DataFrame([
        {
            "check": "one_division_per_parent",
            "passed": not selected["parent_track_id"].duplicated().any(),
            "details": f"{selected['parent_track_id'].nunique()} parents",
        },
        {
            "check": "one_parent_per_daughter",
            "passed": not daughter_ids.duplicated().any(),
            "details": f"{len(daughter_ids)} daughter assignments",
        },
        {
            "check": "two_distinct_daughters",
            "passed": bool(
                (
                    selected["daughter_track_id_a"]
                    != selected["daughter_track_id_b"]
                ).all()
            ),
            "details": "All daughter pairs differ",
        },
    ])

    if not checks["passed"].all():
        raise ValueError("Chapter 39 output validation failed.")

    return checks


validation_df = validate_selected(learned_selected_df)

print("Chapter 39 validation passed.")
display(validation_df)


Chapter 39 validation passed.


,check,passed,details
0,one_division_per_parent,True,18 parents
1,one_parent_per_daughter,True,36 daughter assignments
2,two_distinct_daughters,True,All daughter pairs differ


## 15. Save Chapter 39 outputs

In [19]:
from pathlib import Path
import json
import joblib
import pandas as pd

# ------------------------------------------------------------
# Chapter 39 output directory
# ------------------------------------------------------------

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Saving Chapter 39 outputs to: {OUTPUT_DIR}")

# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

output_paths = {
    "labeled_candidates": (
        OUTPUT_DIR
        / "chapter39_labeled_division_candidates.csv"
    ),
    "predictions": (
        OUTPUT_DIR
        / "chapter39_division_candidate_predictions.csv"
    ),
    "selected": (
        OUTPUT_DIR
        / "chapter39_learned_selected_divisions.csv"
    ),
    "audit": (
        OUTPUT_DIR
        / "chapter39_label_audit.csv"
    ),
    "cv_predictions": (
        OUTPUT_DIR
        / "chapter39_cv_predictions.csv"
    ),
    "cv_metrics": (
        OUTPUT_DIR
        / "chapter39_cv_metrics.csv"
    ),
    "thresholds": (
        OUTPUT_DIR
        / "chapter39_threshold_search.csv"
    ),
    "metrics": (
        OUTPUT_DIR
        / "chapter39_final_metrics.csv"
    ),
    "importance": (
        OUTPUT_DIR
        / "chapter39_feature_importance.csv"
    ),
    "comparison": (
        OUTPUT_DIR
        / "chapter39_selection_comparison.csv"
    ),
    "validation": (
        OUTPUT_DIR
        / "chapter39_output_validation.csv"
    ),
    "features": (
        OUTPUT_DIR
        / "chapter39_feature_columns.json"
    ),
    "model": (
        OUTPUT_DIR
        / "chapter39_division_model.joblib"
    ),
}

# ------------------------------------------------------------
# Helper: always save a CSV, even when the DataFrame is empty
# ------------------------------------------------------------

def save_dataframe(
    dataframe: pd.DataFrame,
    path: Path,
    fallback_columns=None,
) -> None:
    if dataframe is None:
        dataframe = pd.DataFrame(
            columns=fallback_columns or []
        )

    dataframe.to_csv(
        path,
        index=False,
    )

    print(
        f"Saved: {path.name} "
        f"({len(dataframe):,} rows, "
        f"{len(dataframe.columns):,} columns)"
    )


# ------------------------------------------------------------
# Save core outputs
# ------------------------------------------------------------

save_dataframe(
    labeled_df,
    output_paths["labeled_candidates"],
)

save_dataframe(
    labeled_df,
    output_paths["predictions"],
)

save_dataframe(
    learned_selected_df,
    output_paths["selected"],
    fallback_columns=[
        "parent_track_id",
        "daughter_track_id_a",
        "daughter_track_id_b",
        "division_score",
        "learned_division_probability",
        "division_label",
    ],
)

save_dataframe(
    audit_df,
    output_paths["audit"],
)

save_dataframe(
    final_metrics_df,
    output_paths["metrics"],
)

save_dataframe(
    comparison_df,
    output_paths["comparison"],
)

save_dataframe(
    validation_df,
    output_paths["validation"],
)

# ------------------------------------------------------------
# Save optional diagnostic tables
# ------------------------------------------------------------

save_dataframe(
    cv_predictions_df,
    output_paths["cv_predictions"],
    fallback_columns=[
        "parent_track_id",
        "daughter_track_id_a",
        "daughter_track_id_b",
        "division_label",
        "division_probability",
        "fold",
    ],
)

save_dataframe(
    cv_metrics_df,
    output_paths["cv_metrics"],
    fallback_columns=[
        "fold",
        "precision",
        "recall",
        "f1",
        "average_precision",
        "roc_auc",
    ],
)

save_dataframe(
    threshold_search_df,
    output_paths["thresholds"],
    fallback_columns=[
        "threshold",
        "precision",
        "recall",
        "f1",
    ],
)

save_dataframe(
    feature_importance_df,
    output_paths["importance"],
    fallback_columns=[
        "feature",
        "importance",
    ],
)

# ------------------------------------------------------------
# Save feature configuration
# ------------------------------------------------------------

feature_configuration = {
    "feature_columns": FEATURE_COLUMNS,
    "probability_threshold": float(
        selected_threshold
    ),
    "prediction_source": prediction_source,
    "model_name": model_name,
    "supervised_model_trained": (
        final_model is not None
    ),
    "positive_examples": int(
        positive_count
    ),
    "negative_examples": int(
        negative_count
    ),
}

with output_paths["features"].open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_configuration,
        file,
        indent=2,
    )

print(
    f"Saved: {output_paths['features'].name}"
)

# ------------------------------------------------------------
# Save model only when one was actually trained
# ------------------------------------------------------------

if final_model is not None:
    joblib.dump(
        final_model,
        output_paths["model"],
    )

    print(
        f"Saved: {output_paths['model'].name}"
    )
else:
    print(
        "Model file not created because supervised "
        "training did not occur."
    )

# ------------------------------------------------------------
# Create one ZIP containing all Chapter 39 files
# ------------------------------------------------------------

import shutil

zip_base = OUTPUT_DIR / "chapter39_outputs"

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=OUTPUT_DIR,
    base_dir=".",
)

print()
print(f"Created ZIP: {zip_path}")

# ------------------------------------------------------------
# Verify output files
# ------------------------------------------------------------

created_files = sorted(
    OUTPUT_DIR.glob("chapter39_*")
)

verification_rows = []

for path in created_files:
    verification_rows.append(
        {
            "filename": path.name,
            "size_bytes": path.stat().st_size,
            "exists": path.exists(),
        }
    )

output_verification_df = pd.DataFrame(
    verification_rows
)

print()
print(
    f"Chapter 39 files created: "
    f"{len(output_verification_df)}"
)

display(output_verification_df)

Saving Chapter 39 outputs to: /kaggle/working
Saved: chapter39_labeled_division_candidates.csv (93 rows, 35 columns)
Saved: chapter39_division_candidate_predictions.csv (93 rows, 35 columns)
Saved: chapter39_learned_selected_divisions.csv (18 rows, 35 columns)
Saved: chapter39_label_audit.csv (6 rows, 2 columns)
Saved: chapter39_final_metrics.csv (1 rows, 10 columns)
Saved: chapter39_selection_comparison.csv (5 rows, 2 columns)
Saved: chapter39_output_validation.csv (3 rows, 3 columns)
Saved: chapter39_cv_predictions.csv (0 rows, 0 columns)
Saved: chapter39_cv_metrics.csv (0 rows, 0 columns)
Saved: chapter39_threshold_search.csv (0 rows, 0 columns)
Saved: chapter39_feature_importance.csv (0 rows, 0 columns)
Saved: chapter39_feature_columns.json
Model file not created because supervised training did not occur.

Created ZIP: /kaggle/working/chapter39_outputs.zip

Chapter 39 files created: 13


,filename,size_bytes,exists
0,chapter39_cv_metrics.csv,1,True
1,chapter39_cv_predictions.csv,1,True
2,chapter39_division_candidate_predictions.csv,35456,True
3,chapter39_feature_columns.json,815,True
4,chapter39_feature_importance.csv,1,True
5,chapter39_final_metrics.csv,206,True
6,chapter39_label_audit.csv,154,True
7,chapter39_labeled_division_candidates.csv,35456,True
8,chapter39_learned_selected_divisions.csv,7256,True
9,chapter39_output_validation.csv,168,True


# Chapter 39 complete

You now have a reusable learned division-prediction pipeline:

```text
Division candidates
    ↓
Reliable GT labels
    ↓
Feature matrix
    ↓
Grouped validation
    ↓
LightGBM classifier
    ↓
Probability threshold
    ↓
Non-conflicting lineage predictions
```

## Required outputs

```text
chapter39_division_candidate_predictions.csv
chapter39_learned_selected_divisions.csv
chapter39_feature_columns.json
```

When enough labeled positives exist, also preserve:

```text
chapter39_division_model.joblib
```

The next major data-engineering step is to run Chapters 34–38 over many labeled samples, combine their candidate tables, and retrain this model with substantially more true divisions.
